## **Create-GPF-Library-OSW**

In [1]:
import sys
import os
sys.path.append("../../src/")
import createGPFLibrary
import pandas as pd
import polars as pl
from matplotlib_venn import venn2

#### **Create GPF Library**

Create GPF library using the XGB and the LDA scripts

In [2]:
gpfLib_xgb = createGPFLibrary.run("osw/pyprophet_XGB/merged.parquet")

Reading parquet
Complete parquet loading
Normalizing Intensities ... 


In [3]:
gpfLib_lda = createGPFLibrary.run("osw/pyprophet_LDA/merged.parquet")

Reading parquet
Complete parquet loading
Normalizing Intensities ... 


#### **XGBoost Library is slightly bigger than LDA library**

In [4]:
gpfLib_lda = gpfLib_lda.sort_values(by=['ModifiedPeptideSequence', 'PrecursorCharge'])
gpfLib_lda['Precursor'] = gpfLib_lda['ModifiedPeptideSequence'] + gpfLib_lda['PrecursorCharge'].astype(str)
gpfLib_lda['Precursor'] = gpfLib_lda['Precursor'].str.replace(r'^\.', '', regex=True)

gpfLib_lda['Precursor'].drop_duplicates()

7166495      (UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVRK3
7520411           (UniMod:1)AAAAAAAVAGVGRGGGGAEPR2
7283436           (UniMod:1)AAAAAAGAASGLPGPVAQGLK2
4267839                  (UniMod:1)AAAAAAGAGPEMVR2
6888917    (UniMod:1)AAAAAAGPSPGSGPGDSPEGPEGEAPER3
                            ...                   
7669200                YYYAVVDC(UniMod:4)DSPETASK2
4491869                      YYYDGDMIC(UniMod:4)K2
2797320                                  YYYIPQYK2
6833771                             YYYVPADFVEYEK2
5461337                               YYYWAVNPQDR2
Name: Precursor, Length: 113260, dtype: object

In [5]:
gpfLib_xgb = gpfLib_xgb.sort_values(by=['ModifiedPeptideSequence', 'PrecursorCharge'])
gpfLib_xgb['Precursor'] = gpfLib_xgb['ModifiedPeptideSequence'] + gpfLib_xgb['PrecursorCharge'].astype(str)
gpfLib_xgb['Precursor'] = gpfLib_xgb['Precursor'].str.replace(r'^\.', '', regex=True)

gpfLib_xgb['Precursor'].drop_duplicates()

7166495      (UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVRK3
7520405           (UniMod:1)AAAAAAAVAGVGRGGGGAEPR2
7283441           (UniMod:1)AAAAAAGAASGLPGPVAQGLK2
3785088                  (UniMod:1)AAAAAAGAGPEMVR2
6888913    (UniMod:1)AAAAAAGPSPGSGPGDSPEGPEGEAPER3
                            ...                   
7669195                YYYAVVDC(UniMod:4)DSPETASK2
4491869                      YYYDGDMIC(UniMod:4)K2
2797320                                  YYYIPQYK2
6833771                             YYYVPADFVEYEK2
5461343                               YYYWAVNPQDR2
Name: Precursor, Length: 116237, dtype: object

Seems that the numbers are pretty similar to before, just a slight reduction ~1K peptide precursors as expected because this is a smaller search space. Note that this is quite a bit smaller than Mahmoud's library but this is ok because ok to have more conservative statistics.

I think that the XGBoost library is still not overfitting though based on the distributions I have looked at.

#### **Export Library**

In [6]:
gpfLib_xgb.to_csv("osw/2025-03-07-OSW-GPF-Lib.tsv", sep='\t')

#### **Create and Export iRT files**

#### **Compare To DIA-NN Refined Library**

To create the iRT files, just take the original iRT files and replace the RT and IM values with the ones from this new library.

In [7]:
linIrt = pd.read_csv("../K562-Exp-Lib-Analysis/formattedLib/2025-03-06-linear-irt.tsv", sep='\t')
nonLinIrt = pd.read_csv("../K562-Exp-Lib-Analysis/formattedLib/2025-03-06-nonLin-iRT.tsv", sep='\t')
linIrt['Precursor'] = linIrt['ModifiedPeptideSequence'] + linIrt['PrecursorCharge'].astype(str)
linIrt['Precursor'] = linIrt['Precursor'].str.replace(r'^\.', '', regex=True)
nonLinIrt['Precursor'] = nonLinIrt['ModifiedPeptideSequence'] + nonLinIrt['PrecursorCharge'].astype(str)

nonLinIrt['Precursor'] = nonLinIrt['Precursor'].str.replace(r'^\.', '', regex=True)

In [8]:
linIrt = gpfLib_xgb[gpfLib_xgb['Precursor'].isin(linIrt['Precursor'])]
nonLinIrt = gpfLib_xgb[gpfLib_xgb['Precursor'].isin(nonLinIrt['Precursor'])]

Only 17K peptide precursors but I think this is ok because retention time is pretty well calibrated.

In [9]:
nonLinIrt['Precursor'].drop_duplicates()

3785088                  (UniMod:1)AAAAAAGAGPEMVR2
6888913    (UniMod:1)AAAAAAGPSPGSGPGDSPEGPEGEAPER3
2413303                    (UniMod:1)AAAAAGTATSQR2
7473575               (UniMod:1)AAAAASASQDELNQLER2
8049396            (UniMod:1)AAAAKPNNLSLVVHGPGDLR2
                            ...                   
1845390                             YYQTIGNHASYYK3
4343440                                YYSFFDLDPK2
3851132                                YYSLDELSEK2
890934                                   YYVLNALK2
2797320                                  YYYIPQYK2
Name: Precursor, Length: 17362, dtype: object

In [10]:
linIrt

,PrecursorMz,ProductMz,ProteinId,ModifiedPeptideSequence,PeptideSequence,PrecursorCharge,PeptideGroupLabel,NormalizedRetentionTime,PrecursorIonMobility,LibraryIntensity,Annotation,Precursor
3785088,642.8219,398.2034,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,54.789634,1.022708,3881.220558,b5^1,(UniMod:1)AAAAAAGAGPEMVR2
3785090,642.8219,816.4033,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,54.789634,1.022708,10000.000000,y8^1,(UniMod:1)AAAAAAGAGPEMVR2
3785091,642.8219,887.4404,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,54.789634,1.022708,6871.497161,y9^1,(UniMod:1)AAAAAAGAGPEMVR2
3785092,642.8219,958.4775,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,54.789634,1.022708,5101.310545,y10^1,(UniMod:1)AAAAAAGAGPEMVR2
3785093,642.8219,1029.5146,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,54.789634,1.022708,2374.229979,y11^1,(UniMod:1)AAAAAAGAGPEMVR2
...,...,...,...,...,...,...,...,...,...,...,...,...
5463843,737.8546,574.3559,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,15.381136,1.008365,2827.643594,y5^1,(UniMod:1)GEEANDDKKPTTK2
5463844,737.8546,623.8173,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,15.381136,1.008365,5522.783505,y11^2,(UniMod:1)GEEANDDKKPTTK2
5463845,737.8546,1029.4484,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,15.381136,1.008365,2812.967599,b9^1,(UniMod:1)GEEANDDKKPTTK2
5463846,737.8546,1046.5477,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,15.381136,1.008365,3151.612666,y9^1,(UniMod:1)GEEANDDKKPTTK2


In [11]:
len(linIrt['Precursor'].drop_duplicates())

40

Lengths look reasonable

In [12]:
linIrt.to_csv("osw/2025-03-07-LiniRT.tsv", sep='\t', index=False)
nonLinIrt.to_csv("osw/2025-03-07-nonLinIrt.tsv", sep='\t', index=False)

---

---

#### **Create GPF Library Only Filtering**

Create the library with only filtering thus there is no calibration for IM, RT or intensity

In [13]:
gpfLib_xgb = createGPFLibrary.run("osw/pyprophet_XGB/merged.parquet", noIMCalibration=True, noRTCalibration=True, noIntensityCalibration=True)

No RT Calibration
No IM Calibration
No Intensity Calibration
Reading parquet
Complete parquet loading


In [14]:
gpfLib_xgb = gpfLib_xgb.sort_values(by=['ModifiedPeptideSequence', 'PrecursorCharge'])
gpfLib_xgb['Precursor'] = gpfLib_xgb['ModifiedPeptideSequence'] + gpfLib_xgb['PrecursorCharge'].astype(str)
gpfLib_xgb['Precursor'] = gpfLib_xgb['Precursor'].str.replace(r'^\.', '', regex=True)

gpfLib_xgb['Precursor'].drop_duplicates()

7166495      (UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVRK3
7520405           (UniMod:1)AAAAAAAVAGVGRGGGGAEPR2
7283441           (UniMod:1)AAAAAAGAASGLPGPVAQGLK2
3785088                  (UniMod:1)AAAAAAGAGPEMVR2
6888913    (UniMod:1)AAAAAAGPSPGSGPGDSPEGPEGEAPER3
                            ...                   
7669195                YYYAVVDC(UniMod:4)DSPETASK2
4491869                      YYYDGDMIC(UniMod:4)K2
2797320                                  YYYIPQYK2
6833771                             YYYVPADFVEYEK2
5461343                               YYYWAVNPQDR2
Name: Precursor, Length: 117502, dtype: object

Note that the library is slightly bigger than above because not excluding peptide precursors which have less than `min` fragments (since no intensity calibration)

#### **Export Library**

In [15]:
gpfLib_xgb.to_csv("osw/2025-06-02-OSW-GPF-Lib-Only-Filter.tsv", sep='\t')

#### **Create and Export iRT files**

To create the iRT files, just take the original iRT files and replace the RT and IM values with the ones from this new library.

In [16]:
linIrt = pd.read_csv("../K562-Exp-Lib-Analysis/formattedLib/2025-03-06-linear-irt.tsv", sep='\t')
nonLinIrt = pd.read_csv("../K562-Exp-Lib-Analysis/formattedLib/2025-03-06-nonLin-iRT.tsv", sep='\t')
linIrt['Precursor'] = linIrt['ModifiedPeptideSequence'] + linIrt['PrecursorCharge'].astype(str)
linIrt['Precursor'] = linIrt['Precursor'].str.replace(r'^\.', '', regex=True)
nonLinIrt['Precursor'] = nonLinIrt['ModifiedPeptideSequence'] + nonLinIrt['PrecursorCharge'].astype(str)

nonLinIrt['Precursor'] = nonLinIrt['Precursor'].str.replace(r'^\.', '', regex=True)

In [17]:
linIrt = gpfLib_xgb[gpfLib_xgb['Precursor'].isin(linIrt['Precursor'])]
nonLinIrt = gpfLib_xgb[gpfLib_xgb['Precursor'].isin(nonLinIrt['Precursor'])]

Only 17K peptide precursors but I think this is ok because retention time is pretty well calibrated.

In [18]:
nonLinIrt['Precursor'].drop_duplicates()

3785088                  (UniMod:1)AAAAAAGAGPEMVR2
6888913    (UniMod:1)AAAAAAGPSPGSGPGDSPEGPEGEAPER3
2413303                    (UniMod:1)AAAAAGTATSQR2
7473575               (UniMod:1)AAAAASASQDELNQLER2
8049396            (UniMod:1)AAAAKPNNLSLVVHGPGDLR2
                            ...                   
1845390                             YYQTIGNHASYYK3
4343440                                YYSFFDLDPK2
3851132                                YYSLDELSEK2
890934                                   YYVLNALK2
2797320                                  YYYIPQYK2
Name: Precursor, Length: 17379, dtype: object

In [19]:
linIrt

,PrecursorMz,ProductMz,ProteinId,ModifiedPeptideSequence,PeptideSequence,PrecursorCharge,PeptideGroupLabel,NormalizedRetentionTime,PrecursorIonMobility,LibraryIntensity,Annotation,Precursor
3785088,642.8219,398.2034,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,64.227409,1.029010,2718.6753,b5^1,(UniMod:1)AAAAAAGAGPEMVR2
3785090,642.8219,816.4033,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,64.227409,1.029010,10000.0000,y8^1,(UniMod:1)AAAAAAGAGPEMVR2
3785091,642.8219,887.4404,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,64.227409,1.029010,7392.3857,y9^1,(UniMod:1)AAAAAAGAGPEMVR2
3785092,642.8219,958.4775,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,64.227409,1.029010,5807.5674,y10^1,(UniMod:1)AAAAAAGAGPEMVR2
3785093,642.8219,1029.5146,P28482,.(UniMod:1)AAAAAAGAGPEMVR,AAAAAAGAGPEMVR,2,.(Acetyl)AAAAAAGAGPEMVR_2,64.227409,1.029010,2919.7285,y11^1,(UniMod:1)AAAAAAGAGPEMVR2
...,...,...,...,...,...,...,...,...,...,...,...,...
5463843,737.8546,574.3559,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,6.830244,1.016639,3399.3682,y5^1,(UniMod:1)GEEANDDKKPTTK2
5463844,737.8546,623.8173,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,6.830244,1.016639,2411.7140,y11^2,(UniMod:1)GEEANDDKKPTTK2
5463845,737.8546,1029.4484,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,6.830244,1.016639,4877.9790,b9^1,(UniMod:1)GEEANDDKKPTTK2
5463846,737.8546,1046.5477,Q92989,.(UniMod:1)GEEANDDKKPTTK,GEEANDDKKPTTK,2,.(Acetyl)GEEANDDKKPTTK_2,6.830244,1.016639,2983.0605,y9^1,(UniMod:1)GEEANDDKKPTTK2


In [20]:
len(linIrt['Precursor'].drop_duplicates())

40

Lengths look reasonable

In [21]:
linIrt.to_csv("osw/2025-06-02-LiniRT-Refine-GPF-onlyFilter.tsv", sep='\t', index=False)
nonLinIrt.to_csv("osw/2025-06-02-nonLinIrt-Refine-GPF-onlyFilter.tsv", sep='\t', index=False)